In [1]:
import sys
sys.path.append('/var/www/python/Prod/nighthawk/')

import pandas as pd
from pathlib import Path
from nighthawk.data import Constraint

CSV_PATH  = Path('/var/www/python/Qingcheng/QCTest/Manual_bidding/update_sheet/Daily Bidding - daily_constraint.csv')
MARKET    = 'SPP'
THRESHOLD = -900
META_COLS = ['location', 'physical_condition', 'outage_name',
             'comment on this constraint', 'start_date', 'end_date',
             'wind', 'reserve_zone']
COL_ORDER = ['bid_date', 'monitored', 'DA_mvalue', 'RT_mvalue',
             'location', 'physical_condition', 'outage_name',
              'start_date', 'end_date',
             'wind', 'load', 'reserve_zone', "today's wind", 'opportunity']


def _fetch_mvalues(dt_str: str) -> pd.DataFrame:
    opex  = MARKET
    mv_rt = Constraint(oops_constraint_num_df=None, market=opex).get_mvalues(
        start_dt=dt_str, end_dt=dt_str, type='RT', granularity='daily')
    mv_da = Constraint(oops_constraint_num_df=None, market=opex).get_mvalues(
        start_dt=dt_str, end_dt=dt_str, type='DA', granularity='daily')

    all_cons = pd.DataFrame({'oops_constraint_num':
        pd.concat([mv_rt['oops_constraint_num'], mv_da['oops_constraint_num']]).unique()})

    det_rt = Constraint(oops_constraint_num_df=all_cons, market=opex).get_constraint_details(da_or_rt='RT')
    det_da = Constraint(oops_constraint_num_df=all_cons, market=opex).get_constraint_details(da_or_rt='DA')
    details = (pd.concat([det_rt, det_da])
               .drop_duplicates('oops_constraint_num')
               [['oops_constraint_num', 'monitored_clean']]
               .rename(columns={'monitored_clean': 'monitored'}))

    da_sum = (mv_da.merge(details, on='oops_constraint_num', how='left')
              .groupby('monitored')['mvalue'].sum().rename('DA_mvalue').reset_index())
    rt_sum = (mv_rt.merge(details, on='oops_constraint_num', how='left')
              .groupby('monitored')['mvalue'].sum().rename('RT_mvalue').reset_index())

    merged = pd.merge(da_sum, rt_sum, on='monitored', how='outer').fillna(0)
    if merged.empty:
        return pd.DataFrame(columns=['monitored', 'DA_mvalue', 'RT_mvalue'])
    merged['monitored'] = merged['monitored'].str.strip()
    # coerce to numeric (column can be object dtype if a side was empty) before rounding
    merged['DA_mvalue'] = pd.to_numeric(merged['DA_mvalue'], errors='coerce').fillna(0).round(0).astype(int)
    merged['RT_mvalue'] = pd.to_numeric(merged['RT_mvalue'], errors='coerce').fillna(0).round(0).astype(int)
    return merged


def _lookup_metadata(monitored_name: str, df: pd.DataFrame) -> dict:
    prior = df[df['monitored'] == monitored_name]
    if prior.empty:
        return {c: '' for c in META_COLS}
    return {c: prior.sort_values('bid_date').iloc[-1].get(c, '') for c in META_COLS}


def _opportunity(monitored_name: str, df: pd.DataFrame, before_dt) -> str:
    prior = df[(df['monitored'] == monitored_name) & (df['bid_date'] < before_dt)]
    if prior.empty:
        return 'new'
    return pd.Timestamp(prior['bid_date'].max()).strftime('%-m/%-d/%Y')


def update_constraints(start_dt: str, end_dt: str, save: bool = True) -> pd.DataFrame:
    df = pd.read_csv(CSV_PATH)
    df.columns = df.columns.str.strip()
    df = df.rename(columns={'constraints': 'monitored', 'constraints ': 'monitored'})
    df['monitored'] = df['monitored'].str.strip()
    # coerce bad/missing-year bid_dates (e.g. "6/15") to NaT, then drop those rows
    df['bid_date']  = pd.to_datetime(df['bid_date'], format='mixed', errors='coerce')
    df = df.dropna(subset=['bid_date']).reset_index(drop=True)

    date_range = pd.date_range(start=start_dt, end=end_dt, freq='D')
    new_rows   = []

    for dt in date_range:
        dt_str = dt.strftime('%Y-%m-%d')
        print(f"\nFetching {dt_str}...")

        fetched = _fetch_mvalues(dt_str)
        fetched = fetched[
            (fetched['DA_mvalue'] <= THRESHOLD) | (fetched['RT_mvalue'] <= THRESHOLD)
        ].reset_index(drop=True)

        if fetched.empty:
            print(f"  no constraints below threshold")
            continue
        print(f"  {len(fetched)} constraint(s) found")

        existing_mask = df['bid_date'] == dt

        if existing_mask.any():
            for _, frow in fetched.iterrows():
                row_mask = existing_mask & (df['monitored'] == frow['monitored'])
                opp = _opportunity(frow['monitored'], df, before_dt=dt)
                if row_mask.any():
                    df.loc[row_mask, 'DA_mvalue']   = frow['DA_mvalue']
                    df.loc[row_mask, 'RT_mvalue']   = frow['RT_mvalue']
                    df.loc[row_mask, 'opportunity'] = opp
                    print(f"    updated  : {frow['monitored']} (opportunity={opp})")
                else:
                    meta = _lookup_metadata(frow['monitored'], df)
                    new_rows.append({'bid_date': dt, 'monitored': frow['monitored'],
                                     'DA_mvalue': frow['DA_mvalue'], 'RT_mvalue': frow['RT_mvalue'],
                                     **meta, "today's wind": '', 'opportunity': opp})
                    print(f"    appended : {frow['monitored']} (new for this date, opportunity={opp})")
        else:
            for _, frow in fetched.iterrows():
                meta = _lookup_metadata(frow['monitored'], df)
                opp  = _opportunity(frow['monitored'], df, before_dt=dt)
                new_rows.append({'bid_date': dt, 'monitored': frow['monitored'],
                                 'DA_mvalue': frow['DA_mvalue'], 'RT_mvalue': frow['RT_mvalue'],
                                 **meta, "today's wind": '', 'opportunity': opp})
                print(f"    appended : {frow['monitored']} (opportunity={opp})")

    result = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
    result['bid_date'] = pd.to_datetime(result['bid_date'], format='mixed', errors='coerce')
    result = result.dropna(subset=['bid_date']).reset_index(drop=True)

    # Sort: by date first, then within each date by abs(RT - DA) descending
    result['_rank'] = (
        pd.to_numeric(result['RT_mvalue'], errors='coerce').fillna(0) -
        pd.to_numeric(result['DA_mvalue'], errors='coerce').fillna(0)
    ).abs()
    result = (result
              .sort_values(['bid_date', '_rank'], ascending=[True, False])
              .drop(columns='_rank')
              .reset_index(drop=True))

    result['bid_date'] = result['bid_date'].dt.strftime('%-m/%-d/%Y')
    result = result[COL_ORDER]

    print(f"\n{'='*60}")
    print(f"Total rows: {len(result)}  |  New rows added: {len(new_rows)}")
    display(result.tail(len(new_rows) + 3))

    if save:
        result = result.rename(columns={'monitored': 'constraints '})
        result.to_csv(CSV_PATH, index=False)
        print(f"Saved → {CSV_PATH}")

    return result

In [4]:
# today    = pd.Timestamp.now(tz='US/Central').normalize().tz_localize(None)
today = pd.Timestamp('2026-06-29').normalize().tz_localize(None)
start_dt = (today - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
end_dt   = (today + pd.Timedelta(days=1)).strftime('%Y-%m-%d')

result = update_constraints(start_dt, end_dt, save=True)


Fetching 2026-06-28...
  26 constraint(s) found
    updated  : lnadams_i-bvr_crk (opportunity=6/10/2026)
    updated  : lnbeaver1-eurk_spa (opportunity=6/27/2026)
    updated  : lncraig-lenexa (opportunity=new)
    updated  : lncrescent-ctnwd1 (opportunity=new)
    updated  : lndovrt-turcrk2 (opportunity=6/21/2026)
    updated  : lnfirstcrk-roanrdge (opportunity=6/27/2026)
    updated  : lnjagg-pent (opportunity=6/27/2026)
    updated  : lnjay_hawk-frankln5 (opportunity=6/9/2026)
    updated  : lnlar3821-sprgfld (opportunity=6/14/2026)
    updated  : lnmanfl7-clec_pel (opportunity=6/27/2026)
    updated  : lnmonett-aur1241 (opportunity=6/18/2026)
    updated  : lnn345-blackbry (opportunity=6/11/2026)
    updated  : lnosage_og-webbtap4 (opportunity=6/27/2026)
    updated  : lnquail-skyln (opportunity=6/11/2026)
    updated  : lnraun-tekamho (opportunity=6/27/2026)
    updated  : lnrussett-sbrown (opportunity=6/27/2026)
    updated  : lnsugr_crk-sub_h (opportunity=6/24/2026)
    updated

,bid_date,monitored,DA_mvalue,RT_mvalue,location,physical_condition,outage_name,start_date,end_date,wind,load,reserve_zone,today's wind,opportunity
420,6/29/2026,lnmstng1-sw51,-1096.0,-697.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,new
421,6/29/2026,lncrescent-ctnwd1,-1454.0,-1150.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6/28/2026
422,6/29/2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"1: ow: 82, fw: 96, ol: 100, fl: 100\r\n2: ow: ...",NaN
423,6/30/2026,xfmrduncan-duncan,-6804.0,0.0,"South OKGE, Lawton","high load driven, ng generation up, 64kv","Minco - Norman Hills 345kV Line, flow+9mw (con...",2026-06-01 8:00:00,2026-07-02 15:30:00,NaN,NaN,2,,6/29/2026
424,6/30/2026,lnosage_og-webbtap4,-6093.0,0.0,"North Texas, near Tulsa","180kv, high wind driven, Sooner XF 2026-06-19 ...","Morrison - Sooner 138 kV, 5/7-6/30, +42MW",5/7,6/30,high,NaN,"3,4",,6/29/2026
425,6/30/2026,lnweav-tallgras,-5855.0,0.0,"Wichita, Kansas","binds in DA in two days only, flow not close t...",NaN,NaN,NaN,high,NaN,4,,6/29/2026
426,6/30/2026,xfmrnses-nses,-4007.0,0.0,,,,,,,NaN,,,new
427,6/30/2026,lnrussett-sbrown,-3389.0,0.0,South OKGE,"binds high wind, no obvious outage nearby","Pittsburg 345 (PSO) - Valliant 345 345 kV, +15%",2026-06-22 8:00,2026-06-26 16:00,high,NaN,"3,4",,6/29/2026
428,6/30/2026,lncraig-lenexa,-3336.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,6/29/2026
429,6/30/2026,lnwardwa-bismark2,-3054.0,0.0,"Bismark, ND","RT happens when high wind and lwg high at 5, R...","Hay Creek - North Bismark 115 kv, 5/26-6/5, +2...",5/26,6/5/2026 16:00:00,low,NaN,5,,6/27/2026


Saved → /var/www/python/Qingcheng/QCTest/Manual_bidding/update_sheet/Daily Bidding - daily_constraint.csv
